# 06 · Which factor matters, and how well risk can be ranked

**RQ4.** Given route, departure hour and month, how well can delay risk be ranked?

Three models, each with its own honest ceiling: order-dependent sequential R² and conditional drop-one comparisons, a
quantile regression for the tail, and a time-based holdout classifier for risk.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import models, viz

panel = clean.completed(io.load_flights())

ranking = models.incremental_r2(panel)
print(f"model R-squared {ranking.attrs['model_r2']} on {ranking.attrs['rows']:,} sampled flights")
ranking

## 6.1 The tail moves differently

A factor can be irrelevant on average and decisive at the ninetieth percentile, so
a reference-coded full-rank design is fitted to signed arrival delay at q = 0.9. Coefficients compare with recorded reference levels; their magnitudes across different units are not a factor ranking.

In [ ]:
quantile = models.quantile_regression(panel, quantile=0.9)
print(f"pseudo R-squared {quantile['pseudo_r2']}")
quantile["table"].head(8)

## 6.2 Risk classification on a time holdout

Logistic baseline against histogram gradient boosting. The split is chronological, so
no later month trains an earlier one. Both are reported: a reader needs to see how
much the tree model actually adds, and here it is little — five observable factors
rank risk weakly, and that is the finding.

In [ ]:
risk = models.risk_classifier(panel, threshold=60)
print(f"test set starts {risk['split']['actual_test_start']} "
      f"(requested {risk['split']['requested_start']}), "
      f"base rate {risk['base_rate']:.3%}")

pd.DataFrame({name: metrics for name, metrics in risk["models"].items()
              if name != "importance"}).T

In [ ]:
viz.fig_importance(pd.DataFrame(risk["models"]["importance"]))
viz.fig_hour_response(risk["hour_response"], threshold=60)

## Visual audit — Cairo's five questions

### Figure 11 · What the risk model actually uses

| Cairo question | Answer |
|---|---|
| **Guidelines** | Sorted horizontal bars of permutation importance with error bars from five repeats. One visual variable, one uncertainty marker, no colour coding by rank. |
| **Truthful** | The label is permutation importance (AUC drop), not SHAP, and the note says so. Importance is a joint, non-additive association: the bars do not sum to the model's performance and are not read as shares of a cause. |
| **Functional** | The reader sees that departure hour dominates, then month, then carrier, and can stop there. Small factors are still drawn, because dropping them would hide that distance contributes almost nothing. |
| **Beautiful** | Bars ordered by size, category labels at the axis, no legend needed. Error bars thin enough not to read as a second series. |
| **Insightful** | The strongest driver is a field every timetable already publishes. The model's value is therefore not discovery but calibration on a known pattern — which is why the honest conclusion is a negative result, not a predictor. |

### Figure 12 · Modelled risk of arriving 60+ minutes late, by departure hour

| Cairo question | Answer |
|---|---|
| **Guidelines** | A line over 24 ordered hours with the base rate as a reference rule. The reference is what makes "high" mean something. |
| **Truthful** | The base rate (6.1% training, 6.8% test) is drawn as a labelled rule, and hours with very few test flights are not smoothed away. The note states the split dates and that AUC 0.691 is against a 0.50 coin flip. |
| **Functional** | A traveller or planner can read a specific hour's modelled risk and compare it with the base rate in the same glance. |
| **Beautiful** | One line, one rule, one shaded evening band; the axis runs 0–23 in natural order, not sorted by risk. |
| **Insightful** | Risk nearly triples between 06:00 (3.0%) and 16:00 (10.3%), so the useful signal is diurnal and already visible in schedule design — the figure supports the text's claim that the model partly rediscovers the timetable. |